# Module 8 — pandas: Working with Tabular Data

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Create and inspect `Series` and `DataFrame` objects; load data from CSV.
2. Select rows and columns with `[]`, `.loc`, `.iloc` and boolean conditions.
3. Clean data: missing values, duplicates, inconsistent text, wrong types.
4. Transform data: new columns, `map`/`apply`, string and date methods, binning.
5. Summarise data with `value_counts`, `groupby`, `pivot_table`, `crosstab`.
6. Combine tables with `merge` and `concat`; prepare features for scikit-learn.

---

## 8.1 What is pandas?

pandas provides two data structures built on NumPy:

* **`Series`** — a 1-D labelled array (one column).
* **`DataFrame`** — a 2-D table: rows have an **index**, columns have **names**, and each column can have its own dtype.

Think of a DataFrame as a spreadsheet or an SQL table you can program. In ML, the typical workflow is: **load → explore → clean → transform → convert to NumPy → model.**

In [1]:
import numpy as np
import pandas as pd

print(pd.__version__)

3.1.0


## 8.2 Series

In [2]:
s = pd.Series([0.91, 0.85, 0.93], index=["SVM", "kNN", "RF"], name="accuracy")
print(s)
print(s["RF"], s.iloc[0])          # by label, by position
print(s[s > 0.9])                  # boolean filtering, as in NumPy
print(s.mean(), s.idxmax())        # idxmax returns the LABEL of the max
print(s.to_numpy())

SVM    0.91
kNN    0.85
RF     0.93
Name: accuracy, dtype: float64
0.93 0.91
SVM    0.91
RF     0.93
Name: accuracy, dtype: float64
0.8966666666666666 RF
[0.91 0.85 0.93]


## 8.3 Creating DataFrames

From a dictionary of columns, or from a list of records (rows):

In [3]:
df_small = pd.DataFrame({
    "model": ["SVM", "kNN", "RF", "LogReg"],
    "accuracy": [0.91, 0.85, 0.93, 0.88],
    "train_time": [3.2, 0.4, 8.1, 0.9],
})
df_small

,model,accuracy,train_time
0,SVM,0.91,3.2
1,kNN,0.85,0.4
2,RF,0.93,8.1
3,LogReg,0.88,0.9


In [4]:
records = [{"name": "Ana", "grade": 72}, {"name": "Ben", "grade": 65}]
pd.DataFrame(records)

,name,grade
0,Ana,72
1,Ben,65


## 8.4 Loading and inspecting data

We'll use a (synthetic) dataset of MSc students, provided in the `data/` folder. `read_csv` has many options (`sep`, `usecols`, `na_values`, `parse_dates`, …).

In [5]:
df = pd.read_csv("../data/student_performance.csv")
df.head()          # first 5 rows (df.tail() for the last)

,student_id,programme,study_mode,age,hours_studied,attendance_pct,prior_gpa,assignment_score,exam_score,enrolment_date,passed
0,S1000,MSc AI,Full-time,24,15.8,57.6,3.44,50.7,39.5,2025-10-09,0
1,S1001,MSc Data Science,Full-time,24,8.0,92.0,2.94,66.9,32.8,2025-09-20,0
2,S1002,MSc Data Science,Full-time,27,14.9,76.6,NaN,82.2,58.1,2025-09-21,1
3,S1003,MSc AI,Full-time,24,3.5,96.2,3.25,60.6,NaN,2025-09-28,1
4,S1004,MSc AI,Full-time,30,16.2,80.7,3.00,65.0,60.6,2025-10-12,1


In [6]:
print(df.shape)            # (rows, columns)
print(df.columns.tolist())
print(df.dtypes)

(403, 11)
['student_id', 'programme', 'study_mode', 'age', 'hours_studied', 'attendance_pct', 'prior_gpa', 'assignment_score', 'exam_score', 'enrolment_date', 'passed']
student_id              str
programme               str
study_mode              str
age                   int64
hours_studied       float64
attendance_pct      float64
prior_gpa           float64
assignment_score    float64
exam_score          float64
enrolment_date          str
passed                int64
dtype: object


`info()` shows types, non-null counts (missing values!) and memory:

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        403 non-null    str    
 1   programme         403 non-null    str    
 2   study_mode        403 non-null    str    
 3   age               403 non-null    int64  
 4   hours_studied     403 non-null    float64
 5   attendance_pct    386 non-null    float64
 6   prior_gpa         379 non-null    float64
 7   assignment_score  403 non-null    float64
 8   exam_score        391 non-null    float64
 9   enrolment_date    403 non-null    str    
 10  passed            403 non-null    int64  
dtypes: float64(5), int64(2), str(4)
memory usage: 34.8 KB


`describe()` summarises the numeric columns:

In [8]:
df.describe().round(2)

,age,hours_studied,attendance_pct,prior_gpa,assignment_score,exam_score,passed
count,403.00,403.00,386.00,379.00,403.00,391.00,403.00
mean,25.01,12.65,80.65,3.19,65.47,49.91,0.72
std,2.95,5.05,11.40,0.44,10.40,14.12,0.45
min,21.00,1.00,46.10,2.00,33.60,4.90,0.00
25%,23.00,9.30,72.10,2.90,58.55,40.45,0.00
50%,25.00,12.50,80.85,3.19,65.30,50.00,1.00
75%,27.00,16.05,89.25,3.50,72.35,58.55,1.00
max,34.00,26.70,100.00,4.00,94.90,83.20,1.00


In [9]:
df.describe(exclude="number")   # non-numeric (text) columns: count, unique values, most frequent

,student_id,programme,study_mode,enrolment_date
count,403,403,403,403
unique,400,3,4,30
top,S1005,MSc AI,Full-time,2025-09-27
freq,2,176,306,23


**Always look at your data before modelling.** Already we can see: some columns have missing values, and `study_mode` has more unique values than expected.

## 8.5 Selecting data

### Columns

In [10]:
print(df["exam_score"].head(3))                       # one column -> Series
df[["student_id", "programme", "exam_score"]].head(3)  # list of columns -> DataFrame

0    39.5
1    32.8
2    58.1
Name: exam_score, dtype: float64


,student_id,programme,exam_score
0,S1000,MSc AI,39.5
1,S1001,MSc Data Science,32.8
2,S1002,MSc Data Science,58.1


### `.loc` (labels) and `.iloc` (integer positions)

| | selects by | slice end |
|---|---|---|
| `df.loc[rows, cols]` | index labels / column names / boolean masks | **included** |
| `df.iloc[rows, cols]` | integer positions | excluded (like Python) |

In [11]:
print(df.loc[0, "programme"])
df.loc[0:3, ["programme", "exam_score"]]      # rows with labels 0..3 INCLUSIVE

MSc AI


,programme,exam_score
0,MSc AI,39.5
1,MSc Data Science,32.8
2,MSc Data Science,58.1
3,MSc AI,NaN


In [12]:
df.iloc[0:3, 0:4]                             # first 3 rows, first 4 columns

,student_id,programme,study_mode,age
0,S1000,MSc AI,Full-time,24
1,S1001,MSc Data Science,Full-time,24
2,S1002,MSc Data Science,Full-time,27


### Boolean filtering

In [13]:
high_achievers = df[df["exam_score"] >= 80]
print(len(high_achievers))

ai_full_time = df[(df["programme"] == "MSc AI") & (df["study_mode"] == "Full-time")]
print(len(ai_full_time))

subset = df[df["programme"].isin(["MSc AI", "MSc Data Science"])]
print(subset["programme"].unique())

df.query("hours_studied > 25 and attendance_pct > 90")[["student_id", "hours_studied", "attendance_pct"]]

7
143
<StringArray>
['MSc AI', 'MSc Data Science']
Length: 2, dtype: str


,student_id,hours_studied,attendance_pct


Use `.loc` with a mask to filter rows **and** choose columns at once:

In [14]:
df.loc[df["passed"] == 0, ["student_id", "assignment_score", "exam_score"]].head()

,student_id,assignment_score,exam_score
0,S1000,50.7,39.5
1,S1001,66.9,32.8
12,S1012,59.9,41.4
16,S1016,61.2,27.7
17,S1017,54.6,41.9


## 8.6 Cleaning data

Real-world data is messy. Typical problems: inconsistent text, duplicates, missing values, wrong types.

### Inconsistent categories

In [15]:
print(df["study_mode"].value_counts())

study_mode
Full-time    306
Part-time     95
full-time      1
PART-TIME      1
Name: count, dtype: int64


The `.str` accessor applies string methods to a whole column:

In [16]:
df["study_mode"] = df["study_mode"].str.strip().str.lower().str.capitalize()
print(df["study_mode"].value_counts())

study_mode
Full-time    307
Part-time     96
Name: count, dtype: int64


### Duplicates

In [17]:
print("duplicated rows:", df.duplicated().sum())
print("duplicated IDs:", df["student_id"].duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)

duplicated rows: 3
duplicated IDs: 3
(400, 11)


### Missing values

In [18]:
print(df.isna().sum())
print(f"{df.isna().any(axis=1).mean():.1%} of rows have at least one missing value")

student_id           0
programme            0
study_mode           0
age                  0
hours_studied        0
attendance_pct      16
prior_gpa           24
assignment_score     0
exam_score          12
enrolment_date       0
passed               0
dtype: int64
12.2% of rows have at least one missing value


Three main strategies:

1. **Drop** rows/columns (`dropna`) — simple, but loses data.
2. **Impute** with a statistic (`fillna`) — mean, median, mode, or per-group values.
3. Let the model handle it (some models, e.g. gradient boosting, can).

In [19]:
print("rows if we drop all NaNs:", len(df.dropna()))
print("rows if we only require exam_score:", len(df.dropna(subset=["exam_score"])))

rows if we drop all NaNs: 351
rows if we only require exam_score: 388


In [20]:
df_clean = df.copy()
df_clean["attendance_pct"] = df_clean["attendance_pct"].fillna(df_clean["attendance_pct"].median())

# Impute prior_gpa with the median of the student's own programme
df_clean["prior_gpa"] = df_clean["prior_gpa"].fillna(
    df_clean.groupby("programme")["prior_gpa"].transform("median")
)
df_clean = df_clean.dropna(subset=["exam_score"])   # the target-like column: don't invent it
print(df_clean.isna().sum().sum(), "missing values left;", len(df_clean), "rows")

0 missing values left; 388 rows


> ⚠️ In a real ML project, statistics used for imputation must be computed on the **training set only**, then applied to the test set (to avoid data leakage). scikit-learn's `SimpleImputer` does this for you (Module 10).

### Types

In [21]:
df_clean["enrolment_date"] = pd.to_datetime(df_clean["enrolment_date"])
df_clean["programme"] = df_clean["programme"].astype("category")
print(df_clean.dtypes)

student_id                     str
programme                 category
study_mode                     str
age                          int64
hours_studied              float64
attendance_pct             float64
prior_gpa                  float64
assignment_score           float64
exam_score                 float64
enrolment_date      datetime64[us]
passed                       int64
dtype: object


> **Avoid chained assignment** like `df[df.age > 30]["passed"] = 1` — it modifies a temporary copy, not `df` (and in pandas ≥ 3.0 it never works). Always use `df.loc[mask, "column"] = value`.

In [22]:
demo = df_clean.copy()
demo.loc[demo["age"] > 40, "study_mode"] = "Part-time"
print((demo["age"] > 40).sum(), "rows updated")

0 rows updated


## 8.7 Transforming data

### New columns from arithmetic (vectorised)

In [23]:
df_clean["final_score"] = 0.4 * df_clean["assignment_score"] + 0.6 * df_clean["exam_score"]
df_clean["score_gap"] = df_clean["exam_score"] - df_clean["assignment_score"]
df_clean[["assignment_score", "exam_score", "final_score", "score_gap"]].head()

,assignment_score,exam_score,final_score,score_gap
0,50.7,39.5,43.98,-11.2
1,66.9,32.8,46.44,-34.1
2,82.2,58.1,67.74,-24.1
4,65.0,60.6,62.36,-4.4
5,69.4,49.2,57.28,-20.2


### Conditional columns

In [24]:
df_clean["high_attendance"] = np.where(df_clean["attendance_pct"] >= 85, "yes", "no")

df_clean["classification"] = pd.cut(
    df_clean["final_score"],
    bins=[0, 50, 60, 70, np.inf],
    labels=["Fail", "Pass", "Merit", "Distinction"],
    right=False,                     # intervals [0,50), [50,60), ...
    include_lowest=True,
)
df_clean["classification"].value_counts()

classification
Pass           145
Fail           110
Merit           84
Distinction     49
Name: count, dtype: int64

`pd.qcut` bins by **quantiles** instead (equal-sized groups):

In [25]:
df_clean["hours_quartile"] = pd.qcut(df_clean["hours_studied"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
df_clean["hours_quartile"].value_counts().sort_index()

hours_quartile
Q1    100
Q2     97
Q3     94
Q4     97
Name: count, dtype: int64

### `map` and `apply`

* `Series.map(dict_or_function)` — transform each value of one column.
* `DataFrame.apply(function, axis=1)` — apply a function to each **row** (slower; prefer vectorised code when possible).

In [26]:
short = {"MSc AI": "AI", "MSc Data Science": "DS", "MSc Computer Science": "CS"}
df_clean["prog_code"] = df_clean["programme"].map(short)

def risk_flag(row):
    if row["attendance_pct"] < 60 or row["hours_studied"] < 5:
        return "at risk"
    return "ok"

df_clean["risk"] = df_clean.apply(risk_flag, axis=1)
df_clean[["prog_code", "attendance_pct", "hours_studied", "risk"]].head()

,prog_code,attendance_pct,hours_studied,risk
0,AI,57.60,15.8,at risk
1,DS,92.00,8.0,ok
2,DS,76.60,14.9,ok
4,AI,80.70,16.2,ok
5,CS,80.85,10.8,ok


### Dates

The `.dt` accessor exposes date components:

In [27]:
df_clean["enrol_weekday"] = df_clean["enrolment_date"].dt.day_name()
df_clean["days_after_start"] = (df_clean["enrolment_date"] - pd.Timestamp("2025-09-15")).dt.days
df_clean[["enrolment_date", "enrol_weekday", "days_after_start"]].head()

,enrolment_date,enrol_weekday,days_after_start
0,2025-10-09,Thursday,24
1,2025-09-20,Saturday,5
2,2025-09-21,Sunday,6
4,2025-10-12,Sunday,27
5,2025-10-02,Thursday,17


### Renaming, dropping, sorting

In [28]:
tmp = (
    df_clean
    .rename(columns={"hours_studied": "hours"})
    .drop(columns=["score_gap"])
    .sort_values(["programme", "final_score"], ascending=[True, False])
)
tmp[["student_id", "programme", "final_score"]].head()

,student_id,programme,final_score
61,S1061,MSc AI,80.80
124,S1124,MSc AI,79.26
282,S1282,MSc AI,78.70
123,S1123,MSc AI,75.66
19,S1019,MSc AI,75.28


The style above — wrapping a sequence of operations in parentheses, one per line — is called **method chaining** and keeps pandas code readable.

## 8.8 Summarising and grouping

In [29]:
print(df_clean["programme"].value_counts())
print(df_clean["programme"].value_counts(normalize=True).round(3))

programme
MSc AI                  168
MSc Data Science        126
MSc Computer Science     94
Name: count, dtype: int64
programme
MSc AI                  0.433
MSc Data Science        0.325
MSc Computer Science    0.242
Name: proportion, dtype: float64


### `groupby`: split → apply → combine

In [30]:
df_clean.groupby("programme", observed=True)["final_score"].mean().round(2)

programme
MSc AI                  55.99
MSc Computer Science    53.92
MSc Data Science        58.20
Name: final_score, dtype: float64

Several statistics at once, with **named aggregations**:

In [31]:
summary = df_clean.groupby("programme", observed=True).agg(
    n_students=("student_id", "count"),
    mean_final=("final_score", "mean"),
    pass_rate=("passed", "mean"),
    mean_hours=("hours_studied", "mean"),
).round(3)
summary

,n_students,mean_final,pass_rate,mean_hours
programme,,,,
MSc AI,168,55.994,0.720,12.519
MSc Computer Science,94,53.916,0.638,12.361
MSc Data Science,126,58.204,0.770,13.144


Group by several columns:

In [32]:
df_clean.groupby(["programme", "study_mode"], observed=True)["passed"].mean().unstack().round(3)

study_mode,Full-time,Part-time
programme,,
MSc AI,0.741,0.636
MSc Computer Science,0.706,0.462
MSc Data Science,0.813,0.657


`transform` returns a result **aligned with the original rows** — e.g. each student's score relative to their programme average:

In [33]:
df_clean["vs_programme_mean"] = (
    df_clean["final_score"] - df_clean.groupby("programme", observed=True)["final_score"].transform("mean")
)
df_clean[["programme", "final_score", "vs_programme_mean"]].head()

,programme,final_score,vs_programme_mean
0,MSc AI,43.98,-12.014286
1,MSc Data Science,46.44,-11.764444
2,MSc Data Science,67.74,9.535556
4,MSc AI,62.36,6.365714
5,MSc Computer Science,57.28,3.363830


### Pivot tables and cross-tabulations

In [34]:
pd.pivot_table(df_clean, values="final_score", index="programme", columns="study_mode",
               aggfunc="mean", observed=True).round(1)

study_mode,Full-time,Part-time
programme,,
MSc AI,56.8,52.6
MSc Computer Science,55.3,50.3
MSc Data Science,59.6,54.7


In [35]:
pd.crosstab(df_clean["programme"], df_clean["classification"], normalize="index").round(2)

classification,Fail,Pass,Merit,Distinction
programme,,,,
MSc AI,0.28,0.38,0.24,0.11
MSc Computer Science,0.36,0.36,0.19,0.09
MSc Data Science,0.23,0.38,0.21,0.18


### Correlation

In [36]:
num_cols = ["hours_studied", "attendance_pct", "prior_gpa", "assignment_score", "exam_score"]
df_clean[num_cols].corr().round(2)

,hours_studied,attendance_pct,prior_gpa,assignment_score,exam_score
hours_studied,1.00,-0.08,0.04,0.62,0.54
attendance_pct,-0.08,1.00,0.05,0.22,0.26
prior_gpa,0.04,0.05,1.00,0.13,0.25
assignment_score,0.62,0.22,0.13,1.00,0.46
exam_score,0.54,0.26,0.25,0.46,1.00


## 8.9 Combining DataFrames

`merge` joins tables on a key column, like SQL `JOIN`:

In [37]:
programme_info = pd.DataFrame({
    "programme": ["MSc AI", "MSc Data Science", "MSc Computer Science", "MSc Cyber Security"],
    "department": ["Computer Science", "Mathematics", "Computer Science", "Computer Science"],
    "credits": [180, 180, 180, 180],
})

merged = df_clean.astype({"programme": "str"}).merge(programme_info, on="programme", how="left")
merged[["student_id", "programme", "department"]].head()

,student_id,programme,department
0,S1000,MSc AI,Computer Science
1,S1001,MSc Data Science,Mathematics
2,S1002,MSc Data Science,Mathematics
3,S1004,MSc AI,Computer Science
4,S1005,MSc Computer Science,Computer Science


| `how=` | keeps |
|---|---|
| `"inner"` | only keys present in **both** tables (default) |
| `"left"` | all rows of the left table |
| `"right"` | all rows of the right table |
| `"outer"` | all keys from both |

`concat` stacks DataFrames vertically (more rows) or horizontally (more columns):

In [38]:
part1 = df_small.iloc[:2]
part2 = df_small.iloc[2:]
pd.concat([part1, part2], ignore_index=True)

,model,accuracy,train_time
0,SVM,0.91,3.2
1,kNN,0.85,0.4
2,RF,0.93,8.1
3,LogReg,0.88,0.9


## 8.10 From pandas to machine learning

Models need **numbers**. Categorical columns are commonly converted with **one-hot encoding**:

In [39]:
features = df_clean[["hours_studied", "attendance_pct", "prior_gpa", "programme", "study_mode"]]
X_df = pd.get_dummies(features, columns=["programme", "study_mode"], drop_first=True, dtype=int)
X_df.head()

,hours_studied,attendance_pct,prior_gpa,programme_MSc Computer Science,programme_MSc Data Science,study_mode_Part-time
0,15.8,57.60,3.44,0,0,0
1,8.0,92.00,2.94,0,1,0
2,14.9,76.60,3.17,0,1,0
4,16.2,80.70,3.00,0,0,0
5,10.8,80.85,2.80,1,0,0


Then extract NumPy arrays for scikit-learn (which also accepts DataFrames directly):

In [40]:
X = X_df.to_numpy(dtype=float)
y = df_clean["passed"].to_numpy()
print(X.shape, y.shape, X.dtype)

(388, 6) (388,) float64


## 8.11 Saving data

In [41]:
df_clean.to_csv("../data/student_performance_clean.csv", index=False)   # index=False: don't write row numbers
print(pd.read_csv("../data/student_performance_clean.csv").shape)

(388, 21)


Other formats: `to_excel` / `read_excel` (needs `openpyxl`), `to_parquet` / `read_parquet` (fast, typed, compressed; recommended for large data), `to_json`, `to_sql`.

---

## Summary

* `pd.read_csv` → `head`, `info`, `describe`, `value_counts` to explore.
* `df["col"]`, `df.loc[rows, cols]` (labels, end inclusive), `df.iloc[...]` (positions), masks for filtering.
* Clean with `.str` methods, `drop_duplicates`, `isna`/`fillna`/`dropna`, `astype`, `pd.to_datetime`.
* Transform with vectorised arithmetic, `np.where`, `pd.cut`, `map`, `apply`.
* Summarise with `groupby().agg()`, `transform`, `pivot_table`, `crosstab`, `corr`.
* `merge` joins; `concat` stacks; `get_dummies` one-hot encodes; `.to_numpy()` hands data to ML.

---

## Exercises

All exercises use `../data/student_performance.csv`. Solutions: `solutions/08_pandas_solutions.ipynb`.

**Exercise 8.1 — First look.**
Load the data into `students`. Print: (a) the number of rows and columns; (b) the names and dtypes of the columns; (c) the number of missing values per column; (d) the mean, minimum and maximum `age`; (e) how many students are in each programme.

In [42]:
import pandas as pd
import numpy as np
# Your code here

**Exercise 8.2 — Selection.**
(a) Show `student_id`, `age` and `exam_score` for the 5 oldest students. (b) How many MSc AI students studied more than 20 hours a week? (c) What is the average exam score of students with attendance below 70%? (d) Using `.iloc`, show rows 10–14 and the first three columns.

In [43]:
# Your code here

**Exercise 8.3 — Cleaning.**
Create a cleaned copy `clean` that: (a) fixes the inconsistent `study_mode` values; (b) removes duplicate rows; (c) fills missing `attendance_pct` with the **mean attendance of the same study mode**; (d) fills missing `prior_gpa` with the overall median; (e) drops rows with missing `exam_score`. Print the shape and confirm there are no missing values left.

In [44]:
# Your code here

**Exercise 8.4 — Feature engineering.**
On `clean`: (a) add `final_score = 0.4*assignment + 0.6*exam`; (b) add `study_intensity` with values `"low"` (< 10 h), `"medium"` (10–20 h) and `"high"` (> 20 h) using `pd.cut`; (c) add `enrol_week`, the week number of the enrolment relative to 15 Sep 2025 (week 1 = days 0–6).

In [45]:
# Your code here

**Exercise 8.5 — Grouping.**
(a) Pass rate per `study_intensity`. Does studying more help? (b) For each programme, the mean and standard deviation of `final_score` and the number of students. (c) The top 3 students (by `final_score`) **in each programme** (hint: sort, then `groupby(...).head(3)`).

In [46]:
# Your code here

**Exercise 8.6 — Pivot and correlation.**
(a) Build a pivot table of the mean `exam_score` with programmes as rows and `study_intensity` as columns. (b) Which numeric feature has the strongest correlation with `final_score`?

In [47]:
# Your code here

**Exercise 8.7 — Merging and preparing for ML (challenge).**
(a) Create a DataFrame `fees` with columns `programme` and `annual_fee_gbp` (choose values for the three programmes) and merge it into `clean`. (b) Build a feature matrix `X` with `hours_studied`, `attendance_pct`, `prior_gpa`, `age` and one-hot-encoded `programme` and `study_mode`, and target `y = passed`. Print their shapes and the column names of `X`.

In [48]:
# Your code here